#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

In [1]:
# # 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
# import sys
# if 'google.colab' in sys.modules:
#     !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
#     !sudo apt-get -qq -y install fonts-nanum
#     import matplotlib.font_manager as fm
#     font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
#     for fpath in font_files:
#         fm.fontManager.addfont(fpath)

In [2]:
# # 나눔고딕으로 폰트 변경
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'NanumGothic'

In [3]:
# # matplot의 폰트와 폰트 크기 확인
# print(plt.rcParams['font.family'], plt.rcParams['font.size'])

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
# from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV ,cross_validate
from sklearn.metrics import *
import joblib

* 함수 생성

In [5]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [6]:
data = pd.read_csv('data01_train.csv')

#### 2) 기본 정보 조회

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [8]:
data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [9]:
data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [11]:
data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [12]:
data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다.


In [13]:
data.drop('subject', axis=1, inplace=True)

In [14]:
data.head(3)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING


In [15]:
sum(data.isna().sum())

0

### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [16]:
X = data.drop('Activity', axis=1)
y = data.loc[:, 'Activity']

In [17]:
y.unique()

array(['STANDING', 'LAYING', 'WALKING', 'WALKING_DOWNSTAIRS',
       'WALKING_UPSTAIRS', 'SITTING'], dtype=object)

### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [18]:
# 스케일링 필요 없다고 판단

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [19]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다.
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [ ]:
acc_df = pd.DataFrame({'모델': [],
                       'gs_정확도(교차검증)': [],
                       'gs_정확도(테스트)': [],
                       'gs_정확도(fi_교차검증)': [],
                       'gs_정확도(fi_테스트)': []
                      })
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)


### (1) 알고리즘1 : RandomForest

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'n_estimators': [90, 100, 120],
          'max_depth': [4, 6, 8]
          }

In [ ]:
gs = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [90, 100, 120]})

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8, 'n_estimators': 100}
0.967443876888957


In [ ]:
rf_model = gs.best_estimator_
y_pred = rf_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9643059490084985
[[334   0   0   0   0   0]
 [  0 292  18   0   0   0]
 [  0  22 291   0   0   0]
 [  0   0   0 287   3   2]
 [  0   0   0   3 231   5]
 [  0   0   0   3   7 267]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.93      0.94      0.94       310
          STANDING       0.94      0.93      0.94       313
           WALKING       0.98      0.98      0.98       292
WALKING_DOWNSTAIRS       0.96      0.97      0.96       239
  WALKING_UPSTAIRS       0.97      0.96      0.97       277

          accuracy                           0.96      1765
         macro avg       0.96      0.96      0.96      1765
      weighted avg       0.96      0.96      0.96      1765



In [ ]:
joblib.dump(rf_model, 'rf_model_gs.pkl')

['rf_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'RandomForest', 'gs_정확도(교차검증)': gs.best_score_, 'gs_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,NaN,NaN


### (2) 알고리즘2 : LogisticRegressor

In [ ]:
params= [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]},
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]}
]

In [ ]:
gs = GridSearchCV(LogisticRegression(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid=[{'C': [0.1, 0.5, 1, 2, 5, 10, 100],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.1, 0.5, 1, 2, 5, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}])

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 5, 'penalty': 'l1', 'solver': 'liblinear'}
0.9837213486062122


In [ ]:
lr_model = gs.best_estimator_
y_pred = lr_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9847025495750709
[[334   0   0   0   0   0]
 [  0 301   9   0   0   0]
 [  0  16 297   0   0   0]
 [  0   0   0 291   0   1]
 [  0   0   0   0 238   1]
 [  0   0   0   0   0 277]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.95      0.97      0.96       310
          STANDING       0.97      0.95      0.96       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       0.99      1.00      1.00       277

          accuracy                           0.98      1765
         macro avg       0.99      0.99      0.99      1765
      weighted avg       0.98      0.98      0.98      1765



In [ ]:
joblib.dump(lr_model, 'lr_model_gs.pkl')

['lr_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', 'gs_정확도(교차검증)': gs.best_score_, 'gs_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,NaN,NaN
1,LogisticRegression,0.983721,0.984703,NaN,NaN


### (3) 알고리즘3 : SVM

In [ ]:
params = {'kernel': ['linear', 'rbf', 'sigmoid'],
          'C':  [0.01, 0.1, 1, 10, 100, 1000],
          }

In [ ]:
gs = GridSearchCV(SVC(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'kernel': ['linear', 'rbf', 'sigmoid']})

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 100, 'kernel': 'rbf'}
0.9842067854994161


In [ ]:
svm_model = gs.best_estimator_
y_pred = svm_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9886685552407932
[[334   0   0   0   0   0]
 [  0 303   7   0   0   0]
 [  0  11 302   0   0   0]
 [  0   0   0 291   0   1]
 [  0   0   0   0 239   0]
 [  0   0   0   1   0 276]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.96      0.98      0.97       310
          STANDING       0.98      0.96      0.97       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       1.00      1.00      1.00       277

          accuracy                           0.99      1765
         macro avg       0.99      0.99      0.99      1765
      weighted avg       0.99      0.99      0.99      1765



In [ ]:
joblib.dump(svm_model, 'svm_model_gs.pkl')

['svm_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'SVM', 'gs_정확도(교차검증)': gs.best_score_, 'gs_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,NaN,NaN
1,LogisticRegression,0.983721,0.984703,NaN,NaN
2,SVM,0.984207,0.988669,NaN,NaN


### (4) 알고리즘4 : DecisionTreeClassifier

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'max_depth': [2, 4, 6, 8, 10],
          }

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, 10]})

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 10}
0.9300283712206113


In [ ]:
dt_model = gs.best_estimator_
y_pred = dt_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9325779036827195
[[334   0   0   0   0   0]
 [  0 289  21   0   0   0]
 [  0  29 284   0   0   0]
 [  0   0   0 281   4   7]
 [  0   0   0  12 215  12]
 [  0   0   3  13  18 243]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.91      0.93      0.92       310
          STANDING       0.92      0.91      0.91       313
           WALKING       0.92      0.96      0.94       292
WALKING_DOWNSTAIRS       0.91      0.90      0.90       239
  WALKING_UPSTAIRS       0.93      0.88      0.90       277

          accuracy                           0.93      1765
         macro avg       0.93      0.93      0.93      1765
      weighted avg       0.93      0.93      0.93      1765



In [ ]:
joblib.dump(dt_model, 'dt_model_gs.pkl')

['dt_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', 'gs_정확도(교차검증)': gs.best_score_, 'gs_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,NaN,NaN
1,LogisticRegression,0.983721,0.984703,NaN,NaN
2,SVM,0.984207,0.988669,NaN,NaN
3,DecisionTreeClassifier,0.930028,0.932578,NaN,NaN


In [ ]:
acc_df.to_csv('모델 정확도(그리드서치).csv', index=False, encoding='utf-8-sig')

### (5) 알고리즘5 : XGBoost

In [ ]:
# # XGBoost는 label로 숫자를 원함.
# le = LabelEncoder()
# y_xgb_train = le.fit_transform(y_train)

In [ ]:
# XGB_model = XGBClassifier(random_state=42)
# scores = cross_validate(XGB_model, X_train, y_xgb_train, cv=5)
# print(np.mean(scores['test_score']))

In [ ]:
# XGB_model = XGBClassifier(random_state=42)
# XGB_model.fit(X_train, y_xgb_train)
# y_pred = XGB_model.predict(X_test)
# y_pred = le.inverse_transform(y_pred)

# print('accuracy :',accuracy_score(y_test, y_pred))
# print('='*60)
# print(confusion_matrix(y_test, y_pred))
# print('='*60)
# print(classification_report(y_test, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'XGBoost', 'accuracy': accuracy_score(y_test, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

### (6) 알고리즘6 : LightGBM

In [ ]:
# X_train_lgb = X_train.copy()
# X_train_lgb.columns = [f'feature_{i}' for i in range(X_train.shape[1])]

In [ ]:
# X_test_lgb = X_test.copy()
# X_test_lgb.columns = [f'feature_{i}' for i in range(X_test.shape[1])]

In [ ]:
# le = LabelEncoder()
# y_train_lgb = le.fit_transform(y_train)

In [ ]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# scores = cross_validate(LGB_model, X_train_lgb, y_train_lgb, cv=5)
# print(np.mean(scores['test_score']))

In [ ]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# LGB_model.fit(X_train_lgb, y_train_lgb)  # 경고문이 나옴. 너무 많은 특성 중 쓸모 없는 게 많을 때 이런 경고가 뜰 수 있음. 특성 중요도로 피처 수를 조절할 필요가 있음.

# y_pred = LGB_model.predict(X_test)
# y_pred = le.inverse_transform(y_pred)

# print('accuracy :',accuracy_score(y_test, y_pred))
# print('='*60)
# print(confusion_matrix(y_test, y_pred))
# print('='*60)
# print(classification_report(y_test, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'LightGBM', 'accuracy': accuracy_score(y_test, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

### (7) 알고리즘7 : ANN

In [ ]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Input, Dense, Dropout
# model = Sequential()

In [ ]:
# model.add(Input(shape=(561,)))
# model.add(Dense(283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=6, activation='softmax'))

In [ ]:
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# model.fit(X_train, y_train_encoded, batch_size = 32, epochs = 50)

In [ ]:
# y_pred = model.predict(X_test)
# y_pred = np.argmax(y_pred, axis=1)

In [ ]:
# y_pred = le.inverse_transform(y_pred)

# print('accuracy :',accuracy_score(y_test, y_pred))
# print('='*60)
# print(confusion_matrix(y_test, y_pred))
# print('='*60)
# print(classification_report(y_test, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'ANN', 'accuracy': accuracy_score(y_test, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

## (옵션) 변수 중요도 상위 N개를 선정하여 모델링

### 방법1. 랜덤 포레스트에서 중요도 상위 변수들을 채택

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,NaN,NaN
1,LogisticRegression,0.983721,0.984703,NaN,NaN
2,SVM,0.984207,0.988669,NaN,NaN
3,DecisionTreeClassifier,0.930028,0.932578,NaN,NaN


In [ ]:
RFmodel_fi_df = plot_feature_importance(rf_model.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,tGravityAcc-mean()-X,0.025189
1,fBodyAccJerk-max()-X,0.024606
2,tBodyGyroJerk-iqr()-Z,0.024346
3,tGravityAcc-energy()-X,0.023527
4,tGravityAcc-max()-X,0.023322
...,...,...
556,"tBodyGyroJerk-arCoeff()-Y,2",0.000020
557,"tBodyAccJerk-arCoeff()-Y,3",0.000017
558,"tBodyAcc-arCoeff()-Y,3",0.000015
559,"fBodyAcc-bandsEnergy()-33,48.1",0.000012


In [ ]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [ ]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [ ]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features)}")

누적 중요도 95%까지 선택된 변수 개수: 222


In [ ]:
X_fi = data[RFmodel_fi_df['feature_name'][:300]]
y_fi = data.loc[:, 'Activity']

In [ ]:
X_train_fi, X_val_fi, y_train_fi, y_val_fi = train_test_split(X_fi, y_fi, test_size=0.3, random_state=42)

In [ ]:
# X_test_fi = X_test[RFmodel_fi_df['feature_name'][:300]]

### (1) 랜덤 포레스트

In [ ]:
# feature 개수가 줄었으므로(변경됬으므로) 그리드서치 다시 해보기

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'n_estimators': [90, 100, 120],
          'max_depth': [4, 6, 8]
          }

In [ ]:
gs = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [90, 100, 120]})

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8, 'n_estimators': 90}
0.9645283063384019


In [ ]:
rf_model = gs.best_estimator_
y_pred = rf_model.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.9643059490084985
[[334   0   0   0   0   0]
 [  0 294  16   0   0   0]
 [  0  25 288   0   0   0]
 [  0   0   0 286   3   3]
 [  0   0   0   3 232   4]
 [  0   0   0   3   6 268]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.92      0.95      0.93       310
          STANDING       0.95      0.92      0.93       313
           WALKING       0.98      0.98      0.98       292
WALKING_DOWNSTAIRS       0.96      0.97      0.97       239
  WALKING_UPSTAIRS       0.97      0.97      0.97       277

          accuracy                           0.96      1765
         macro avg       0.96      0.96      0.96      1765
      weighted avg       0.96      0.96      0.96      1765



In [ ]:
joblib.dump(rf_model, 'rf_model_gs_fi.pkl')

['rf_model_gs_fi.pkl']

In [ ]:
acc_df.loc[0, 'gs_정확도(fi_교차검증)'] = gs.best_score_
acc_df.loc[0, 'gs_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306
1,LogisticRegression,0.983721,0.984703,NaN,NaN
2,SVM,0.984207,0.988669,NaN,NaN
3,DecisionTreeClassifier,0.930028,0.932578,NaN,NaN


### (2) LinearRegression

In [ ]:
params= [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]},
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]}
]

In [ ]:
gs = GridSearchCV(LogisticRegression(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid=[{'C': [0.1, 0.5, 1, 2, 5, 10, 100],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.1, 0.5, 1, 2, 5, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}])

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
0.9757024973752196


In [ ]:
lr_model = gs.best_estimator_
y_pred = lr_model.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.9858356940509915
[[334   0   0   0   0   0]
 [  0 304   6   0   0   0]
 [  0  17 296   0   0   0]
 [  0   0   0 291   0   1]
 [  0   0   0   0 239   0]
 [  0   0   0   0   1 276]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.95      0.98      0.96       310
          STANDING       0.98      0.95      0.96       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       1.00      1.00      1.00       277

          accuracy                           0.99      1765
         macro avg       0.99      0.99      0.99      1765
      weighted avg       0.99      0.99      0.99      1765



In [ ]:
joblib.dump(lr_model, 'lr_model_gs_fi.pkl')

['lr_model_gs_fi.pkl']

In [ ]:
acc_df.loc[1, 'gs_정확도(fi_교차검증)'] = gs.best_score_
acc_df.loc[1, 'gs_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836
2,SVM,0.984207,0.988669,NaN,NaN
3,DecisionTreeClassifier,0.930028,0.932578,NaN,NaN


### (3) SVM

In [ ]:
params = {'kernel': ['linear', 'rbf', 'sigmoid'],
          'C':  [0.01, 0.1, 1, 10, 100, 1000],
          }

In [ ]:
gs = GridSearchCV(SVC(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'kernel': ['linear', 'rbf', 'sigmoid']})

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 100, 'kernel': 'rbf'}
0.98566516061296


In [ ]:
svm_model = gs.best_estimator_
y_pred = svm_model.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.9915014164305949
[[334   0   0   0   0   0]
 [  0 306   4   0   0   0]
 [  0  11 302   0   0   0]
 [  0   0   0 292   0   0]
 [  0   0   0   0 239   0]
 [  0   0   0   0   0 277]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.97      0.99      0.98       310
          STANDING       0.99      0.96      0.98       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       1.00      1.00      1.00       277

          accuracy                           0.99      1765
         macro avg       0.99      0.99      0.99      1765
      weighted avg       0.99      0.99      0.99      1765



In [ ]:
joblib.dump(svm_model, 'svm_model_gs_fi.pkl')

['svm_model_gs_fi.pkl']

In [ ]:
acc_df.loc[2, 'gs_정확도(fi_교차검증)'] = gs.best_score_
acc_df.loc[2, 'gs_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836
2,SVM,0.984207,0.988669,0.985665,0.991501
3,DecisionTreeClassifier,0.930028,0.932578,NaN,NaN


### (4) DecisionTreeClassifier

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'max_depth': [2, 4, 6, 8, 10],
          }

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, 10]})

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 10}
0.9365864879849944


In [ ]:
dt_model = gs.best_estimator_
y_pred = dt_model.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.9371104815864023
[[334   0   0   0   0   0]
 [  0 289  21   0   0   0]
 [  0  22 291   0   0   0]
 [  0   0   0 280   4   8]
 [  0   0   0  11 214  14]
 [  0   0   3  15  13 246]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.93      0.93      0.93       310
          STANDING       0.92      0.93      0.93       313
           WALKING       0.92      0.96      0.94       292
WALKING_DOWNSTAIRS       0.93      0.90      0.91       239
  WALKING_UPSTAIRS       0.92      0.89      0.90       277

          accuracy                           0.94      1765
         macro avg       0.94      0.93      0.93      1765
      weighted avg       0.94      0.94      0.94      1765



In [ ]:
joblib.dump(dt_model, 'dt_model_gs_fi.pkl')

['dt_model_gs_fi.pkl']

In [ ]:
acc_df.loc[3, 'gs_정확도(fi_교차검증)'] = gs.best_score_
acc_df.loc[3, 'gs_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836
2,SVM,0.984207,0.988669,0.985665,0.991501
3,DecisionTreeClassifier,0.930028,0.932578,0.936586,0.937110


In [ ]:
acc_df.to_csv('모델 정확도(그리드서치_fi).csv', index=False, encoding='utf-8-sig')

##방법2. PCA 활용

In [20]:
acc_df = pd.read_csv('모델 정확도(그리드서치_fi).csv')
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836
2,SVM,0.984207,0.988669,0.985665,0.991501
3,DecisionTreeClassifier,0.930028,0.932578,0.936586,0.937110


In [21]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)

print("선택된 주성분 수:", pca.n_components_)
print("누적 설명력:", pca.explained_variance_ratio_.cumsum()[-1])

선택된 주성분 수: 66
누적 설명력: 0.9502722599485613


In [22]:
X_val_pca = pca.transform(X_val)

### 랜덤포레스트

In [23]:
params = {'criterion': ['gini', 'entropy'],
          'n_estimators': [90, 100, 120],
          'max_depth': [4, 6, 8]
          }

In [24]:
gs = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [90, 100, 120]})

In [25]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8, 'n_estimators': 100}
0.9139918484351591


In [26]:
rf_model = gs.best_estimator_
y_pred = rf_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9206798866855525
[[334   0   0   0   0   0]
 [  5 257  48   0   0   0]
 [  0  37 276   0   0   0]
 [  0   0   0 279   3  10]
 [  0   0   0   7 219  13]
 [  0   0   0   5  12 260]]
                    precision    recall  f1-score   support

            LAYING       0.99      1.00      0.99       334
           SITTING       0.87      0.83      0.85       310
          STANDING       0.85      0.88      0.87       313
           WALKING       0.96      0.96      0.96       292
WALKING_DOWNSTAIRS       0.94      0.92      0.93       239
  WALKING_UPSTAIRS       0.92      0.94      0.93       277

          accuracy                           0.92      1765
         macro avg       0.92      0.92      0.92      1765
      weighted avg       0.92      0.92      0.92      1765



In [27]:
acc_df.loc[0, 'gs_정확도(pca_교차검증)'] = gs.best_score_
acc_df.loc[0, 'gs_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [28]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트),gs_정확도(pca_교차검증),gs_정확도(pca_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306,0.913992,0.92068
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836,NaN,NaN
2,SVM,0.984207,0.988669,0.985665,0.991501,NaN,NaN
3,DecisionTreeClassifier,0.930028,0.932578,0.936586,0.937110,NaN,NaN


### 로지스틱회귀

In [29]:
params= [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]},
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]}
]

In [30]:
gs = GridSearchCV(LogisticRegression(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid=[{'C': [0.1, 0.5, 1, 2, 5, 10, 100],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.1, 0.5, 1, 2, 5, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}])

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

In [31]:
lr_model = gs.best_estimator_
y_pred = lr_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9711048158640226
[[334   0   0   0   0   0]
 [  3 293  14   0   0   0]
 [  0  23 290   0   0   0]
 [  0   0   0 288   1   3]
 [  0   0   0   0 236   3]
 [  0   0   0   2   2 273]]
                    precision    recall  f1-score   support

            LAYING       0.99      1.00      1.00       334
           SITTING       0.93      0.95      0.94       310
          STANDING       0.95      0.93      0.94       313
           WALKING       0.99      0.99      0.99       292
WALKING_DOWNSTAIRS       0.99      0.99      0.99       239
  WALKING_UPSTAIRS       0.98      0.99      0.98       277

          accuracy                           0.97      1765
         macro avg       0.97      0.97      0.97      1765
      weighted avg       0.97      0.97      0.97      1765



In [32]:
acc_df.loc[1, 'gs_정확도(pca_교차검증)'] = gs.best_score_
acc_df.loc[1, 'gs_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [33]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트),gs_정확도(pca_교차검증),gs_정확도(pca_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306,0.913992,0.920680
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836,0.964528,0.971105
2,SVM,0.984207,0.988669,0.985665,0.991501,NaN,NaN
3,DecisionTreeClassifier,0.930028,0.932578,0.936586,0.937110,NaN,NaN


### SVM

In [34]:
params = {'kernel': ['linear', 'rbf', 'sigmoid'],
          'C':  [0.01, 0.1, 1, 10, 100, 1000],
          }

In [35]:
gs = GridSearchCV(SVC(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'kernel': ['linear', 'rbf', 'sigmoid']})

In [36]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 1000, 'kernel': 'rbf'}
0.9737607498024042


In [37]:
svm_model = gs.best_estimator_
y_pred = svm_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9824362606232294
[[334   0   0   0   0   0]
 [  1 298  11   0   0   0]
 [  0  17 296   0   0   0]
 [  0   0   0 291   0   1]
 [  0   0   0   0 239   0]
 [  0   0   0   1   0 276]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.95      0.96      0.95       310
          STANDING       0.96      0.95      0.95       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       1.00      1.00      1.00       277

          accuracy                           0.98      1765
         macro avg       0.98      0.98      0.98      1765
      weighted avg       0.98      0.98      0.98      1765



In [38]:
acc_df.loc[2, 'gs_정확도(pca_교차검증)'] = gs.best_score_
acc_df.loc[2, 'gs_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [39]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트),gs_정확도(pca_교차검증),gs_정확도(pca_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306,0.913992,0.920680
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836,0.964528,0.971105
2,SVM,0.984207,0.988669,0.985665,0.991501,0.973761,0.982436
3,DecisionTreeClassifier,0.930028,0.932578,0.936586,0.937110,NaN,NaN


### 결정트리

In [40]:
params = {'criterion': ['gini', 'entropy'],
          'max_depth': [2, 4, 6, 8, 10],
          }

In [41]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, 10]})

In [42]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8}
0.8551973598839198


In [43]:
dt_model = gs.best_estimator_
y_pred = dt_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.8572237960339943
[[330   3   0   0   0   1]
 [  6 224  80   0   0   0]
 [  0  43 270   0   0   0]
 [  0   0   0 277   9   6]
 [  2   0   0  20 188  29]
 [  1   1   1  24  26 224]]
                    precision    recall  f1-score   support

            LAYING       0.97      0.99      0.98       334
           SITTING       0.83      0.72      0.77       310
          STANDING       0.77      0.86      0.81       313
           WALKING       0.86      0.95      0.90       292
WALKING_DOWNSTAIRS       0.84      0.79      0.81       239
  WALKING_UPSTAIRS       0.86      0.81      0.83       277

          accuracy                           0.86      1765
         macro avg       0.86      0.85      0.85      1765
      weighted avg       0.86      0.86      0.86      1765



In [44]:
acc_df.loc[3, 'gs_정확도(pca_교차검증)'] = gs.best_score_
acc_df.loc[3, 'gs_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [45]:
acc_df

,모델,gs_정확도(교차검증),gs_정확도(테스트),gs_정확도(fi_교차검증),gs_정확도(fi_테스트),gs_정확도(pca_교차검증),gs_정확도(pca_테스트)
0,RandomForest,0.967444,0.964306,0.964528,0.964306,0.913992,0.920680
1,LogisticRegression,0.983721,0.984703,0.975702,0.985836,0.964528,0.971105
2,SVM,0.984207,0.988669,0.985665,0.991501,0.973761,0.982436
3,DecisionTreeClassifier,0.930028,0.932578,0.936586,0.937110,0.855197,0.857224


In [46]:
acc_df.to_csv('모델 정확도(그리드서치_fi_pca).csv', index=False, encoding='utf-8-sig')